# Paper Reproduction

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV , train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from metric_learn import MLKR
from sklearn.ensemble import RandomForestRegressor

# REu1

In [12]:
# read the CSV file into a DataFrame
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:", df.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# splitting the dataset into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

np.random.seed(42)

test_indices = np.random.choice(
    len(X),
    round(len(X) * 0.3),
    replace=False
)

train_indices = np.array(
    list(set(range(len(X))) - set(test_indices))
)

X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print("DATA PARTITION")
print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"Best k: {best_k}")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original dataset shape: (1311, 122)
DATA PARTITION
Total sources   : 1311
Training sources: 918
Testing sources : 393
Best k: 5
Best CV eta_0.15: 0.08932154801720019
Best k: 5
eta_0.15: 8.14%
eta_2sigma: 6.62%
sigma: 0.1022
sigma_NMAD: 0.0523
R²: 0.6479
MSE: 0.0597
RMSE: 0.2442
MAE: 0.1072


# RMa1

In [13]:
class MahalanobisKNN(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=2):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.covariance_ = np.cov(X, rowvar=False)
        self.VI_ = np.linalg.pinv(self.covariance_)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"VI": self.VI_}
        )

        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)



# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', MahalanobisKNN())
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Best k: 5
Best CV eta_0.15: 0.07953893932154801
eta_0.15: 5.85%
eta_2sigma: 5.09%
sigma: 0.1164
sigma_NMAD: 0.0312
R²: 0.5928
MSE: 0.0690
RMSE: 0.2626
MAE: 0.0891


# RML1

In [14]:
# standardising the input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# MLKR metric learning
mlkr = MLKR()

mlkr.fit(X_train, y_train)

X_train_mlkr = mlkr.transform(X_train)
X_test_mlkr = mlkr.transform(X_test)


# KNN using the learned MLKR metric
knn = KNeighborsRegressor(
    metric="euclidean"
)

param_grid = {
    "n_neighbors": range(2, 20)
}

cv = KFold(n_splits=10, shuffle=True, random_state=10)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    scoring=make_scorer(eta_015, greater_is_better=False),
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_mlkr, y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:", best_cv_eta)


# prediction
y_pred = best_model.predict(X_test_mlkr)


# normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15
eta_015_value = eta_015(y_test, y_pred) * 100

# sigma
sigma_value = np.std(delta_z)

# eta_2sigma
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma_value) * 100

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

# R²
r2 = r2_score(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test, y_pred)


print(f"Best k: {best_k}")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

Best k: 2
Best CV eta_0.15: 0.0686335403726708
Best k: 2
eta_0.15: 6.11%
eta_2sigma: 5.34%
sigma: 0.0948
sigma_NMAD: 0.0442
R²: 0.7808
MSE: 0.0371
RMSE: 0.1927
MAE: 0.0889


## RRF1

In [15]:
# standardising the input photometry
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Random Forest regression
rf = RandomForestRegressor(
    n_estimators=44,
    random_state=42
)

rf.fit(X_train, y_train)

# predicting redshifts
y_pred = rf.predict(X_test)


# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15
eta_015_value = eta_015(y_test, y_pred) * 100

# sigma
sigma_value = np.std(delta_z)

# eta_2sigma
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma_value) * 100

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

# R²
r2 = r2_score(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test, y_pred)


print(f"Trees: 44")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Trees: 44
eta_0.15: 8.14%
eta_2sigma: 4.83%
sigma: 0.1330
sigma_NMAD: 0.0506
R²: 0.5881
MSE: 0.0698
RMSE: 0.2642
MAE: 0.1123


# REu2

In [16]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_elais[features].to_numpy(dtype=np.float32)
y_train = df_elais[target].to_numpy(dtype=np.float32)
X_test = df_cdfs[features].to_numpy(dtype=np.float32)
y_test = df_cdfs[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best k: 5
Best CV eta_0.15: 0.10710204081632653
eta_0.15: 8.70%
eta_2sigma: 5.51%
sigma: 0.1196
sigma_NMAD: 0.0618
R²: 0.6008
MSE: 0.0650
RMSE: 0.2549
MAE: 0.1172


# RMa2

In [17]:
class MahalanobisKNN(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=2):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.covariance_ = np.cov(X, rowvar=False)
        self.VI_ = np.linalg.pinv(self.covariance_)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"VI": self.VI_}
        )

        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)



# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', MahalanobisKNN())
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Best k: 3
Best CV eta_0.15: 0.07069387755102043
eta_0.15: 8.70%
eta_2sigma: 5.39%
sigma: 0.1462
sigma_NMAD: 0.0363
R²: 0.5843
MSE: 0.0676
RMSE: 0.2601
MAE: 0.1021


# RML2

In [18]:
# standardising the input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# MLKR metric learning
mlkr = MLKR()

mlkr.fit(X_train, y_train)

X_train_mlkr = mlkr.transform(X_train)
X_test_mlkr = mlkr.transform(X_test)


# KNN using the learned MLKR metric
knn = KNeighborsRegressor(
    metric="euclidean"
)

param_grid = {
    "n_neighbors": range(2, 20)
}

cv = KFold(n_splits=10, shuffle=True, random_state=10)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    scoring=make_scorer(eta_015, greater_is_better=False),
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_mlkr, y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:", best_cv_eta)


# prediction
y_pred = best_model.predict(X_test_mlkr)


# normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15
eta_015_value = eta_015(y_test, y_pred) * 100

# sigma
sigma_value = np.std(delta_z)

# eta_2sigma
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma_value) * 100

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

# R²
r2 = r2_score(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test, y_pred)


print(f"Best k: {best_k}")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

Best k: 4
Best CV eta_0.15: 0.060734693877551024
Best k: 4
eta_0.15: 7.72%
eta_2sigma: 4.29%
sigma: 0.1240
sigma_NMAD: 0.0454
R²: 0.6738
MSE: 0.0531
RMSE: 0.2304
MAE: 0.0996


# RRF2

In [19]:
# standardising the input photometry
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Random Forest regression
rf = RandomForestRegressor(
    n_estimators=44,
    random_state=42
)

rf.fit(X_train, y_train)

# predicting redshifts
y_pred = rf.predict(X_test)


# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15
eta_015_value = eta_015(y_test, y_pred) * 100

# sigma
sigma_value = np.std(delta_z)

# eta_2sigma
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma_value) * 100

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

# R²
r2 = r2_score(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test, y_pred)


print(f"Trees: 44")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Trees: 44
eta_0.15: 14.58%
eta_2sigma: 5.39%
sigma: 0.2038
sigma_NMAD: 0.0619
R²: 0.2275
MSE: 0.1257
RMSE: 0.3545
MAE: 0.1603


# REu3

In [20]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_cdfs[features].to_numpy(dtype=np.float32)
y_train = df_cdfs[target].to_numpy(dtype=np.float32)
X_test = df_elais[features].to_numpy(dtype=np.float32)
y_test = df_elais[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best k: 11
Best CV eta_0.15: 0.08706714844926226
eta_0.15: 8.28%
eta_2sigma: 4.85%
sigma: 0.1233
sigma_NMAD: 0.0598
R²: 0.5262
MSE: 0.0889
RMSE: 0.2981
MAE: 0.1251


# RMa3

In [21]:
class MahalanobisKNN(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=2):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.covariance_ = np.cov(X, rowvar=False)
        self.VI_ = np.linalg.pinv(self.covariance_)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"VI": self.VI_}
        )

        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)



# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', MahalanobisKNN())
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Best k: 5
Best CV eta_0.15: 0.06991869918699187
eta_0.15: 7.68%
eta_2sigma: 5.45%
sigma: 0.1330
sigma_NMAD: 0.0358
R²: 0.5433
MSE: 0.0857
RMSE: 0.2927
MAE: 0.1079


# RML3

In [22]:
# standardising the input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# MLKR metric learning
mlkr = MLKR()

mlkr.fit(X_train, y_train)

X_train_mlkr = mlkr.transform(X_train)
X_test_mlkr = mlkr.transform(X_test)


# KNN using the learned MLKR metric
knn = KNeighborsRegressor(
    metric="euclidean"
)

param_grid = {
    "n_neighbors": range(2, 20)
}

cv = KFold(n_splits=10, shuffle=True, random_state=10)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    scoring=make_scorer(eta_015, greater_is_better=False),
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_mlkr, y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:", best_cv_eta)


# prediction
y_pred = best_model.predict(X_test_mlkr)


# normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15
eta_015_value = eta_015(y_test, y_pred) * 100

# sigma
sigma_value = np.std(delta_z)

# eta_2sigma
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma_value) * 100

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

# R²
r2 = r2_score(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test, y_pred)


print(f"Best k: {best_k}")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

Best k: 4
Best CV eta_0.15: 0.05761818729298405
Best k: 4
eta_0.15: 9.49%
eta_2sigma: 5.45%
sigma: 0.1297
sigma_NMAD: 0.0475
R²: 0.5787
MSE: 0.0790
RMSE: 0.2811
MAE: 0.1144


# RRF3

In [23]:
# standardising the input photometry
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Random Forest regression
rf = RandomForestRegressor(
    n_estimators=44,
    random_state=42
)

rf.fit(X_train, y_train)

# predicting redshifts
y_pred = rf.predict(X_test)


# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15
eta_015_value = eta_015(y_test, y_pred) * 100

# sigma
sigma_value = np.std(delta_z)

# eta_2sigma
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma_value) * 100

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

# R²
r2 = r2_score(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test, y_pred)


print(f"Trees: 44")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Trees: 44
eta_0.15: 10.71%
eta_2sigma: 4.65%
sigma: 0.1370
sigma_NMAD: 0.0553
R²: 0.5458
MSE: 0.0852
RMSE: 0.2919
MAE: 0.1257


In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import KNeighborsRegressor
from metric_learn import MLKR
import numpy as np


# read the CSV file into a DataFrame
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:", df.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# splitting the dataset into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

np.random.seed(42)

test_indices = np.random.choice(
    len(X),
    round(len(X) * 0.3),
    replace=False
)

train_indices = np.array(
    list(set(range(len(X))) - set(test_indices))
)

X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]


def eta_015(y_true, y_pred):
    dz = np.abs(y_pred - y_true) / (1 + y_true)
    return np.mean(dz > 0.15)

def eta_2sigma(y_true, y_pred):
    dz = np.abs(y_pred - y_true) / (1 + y_true)
    sigma = 1.48 * np.median(np.abs(dz - np.median(dz)))
    return np.mean(dz > 2 * sigma)

def sigma_metric(y_true, y_pred):
    dz = (y_pred - y_true) / (1 + y_true)
    return np.sqrt(np.mean(dz**2))

def sigma_nmad(y_true, y_pred):
    dz = (y_pred - y_true) / (1 + y_true)
    return 1.48 * np.median(np.abs(dz - np.median(dz)))

def run_rml1(X_train, y_train, X_test, y_test):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("mlkr", MLKR()),
        ("knn", KNeighborsRegressor())
    ])

    param_grid = {
        "mlkr__n_neighbors": [3, 5, 10, 15],
        "knn__n_neighbors": [2, 3, 4, 5, 7, 10, 15]
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipe,
        param_grid,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    eta015 = eta_015(y_test, y_pred)
    eta2sig = eta_2sigma(y_test, y_pred)
    sigma = sigma_metric(y_test, y_pred)
    snmad = sigma_nmad(y_test, y_pred)

    ss_res = np.sum((y_test - y_pred) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    r2 = 1 - ss_res / ss_tot

    rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
    mae = np.mean(np.abs(y_test - y_pred))

    print("Best parameters:", grid.best_params_)
    print("Best CV RMSE:", np.sqrt(-grid.best_score_))
    print("Test η_0.15:", eta015)
    print("Test η_2σ:", eta2sig)
    print("Test σ:", sigma)
    print("Test σ_NMAD:", snmad)
    print("Test R²:", r2)
    print("Test RMSE:", rmse)
    print("Test MAE:", mae)

    return {
        "model": best_model,
        "y_pred": y_pred,
        "best_params": grid.best_params_,
        "eta_015": eta015,
        "eta_2sigma": eta2sig,
        "sigma": sigma,
        "sigma_nmad": snmad,
        "r2": r2,
        "rmse": rmse,
        "mae": mae
    }

Original dataset shape: (1311, 122)
